# 02c – Feature sets: with and without heating information and municipality (task 22)

**Question:** how much do the heating information and the municipality add to the ranking of houses?

**Why we ask**: the register shows *today's* heating, while the target covers earlier
years. We dropped the buildings whose heating visibly changed (task 9), but for the undeclared houses, the ones the
office wants to rank, we cannot check the heating at all. So we want to know how much the model depends on it.
The municipality is the largest block of columns (one per municipality), so we also check whether it helps or only adds noise.

**Feature sets**

| Feature set | heating (energy group, heating system) | municipality | expert rule | Note |
|---|---|---|---|---|
| Full | yes | yes | yes | the model of task 20, the reference |
| Full, no expert rule | yes | yes | no | task 20 "expert rule off" |
| No municipality | yes | no | yes | |
| No heating | no | yes | no | the expert rule needs the energy group, so it drops out too |
| No heating, no municipality | no | no | no | only size, construction period and building class |

**Everything else is exactly as in task 20** (`02_estimate.ipynb`): same folds, trained on all buildings, judged on
the houses of the validation fold, the same three models and the same values of alpha. Alpha is chosen again for
every feature set (README decision 16), so no set is held back by a penalty tuned for another one.

Output: `output/tables/task22_feature_sets.csv`, `output/tables/task22_per_fold.csv`,
`output/figures/task22_feature_sets.png`. Run `02_estimate.ipynb` first: section 4 compares with its results.
The notebook takes about 13 minutes, most of it for the lasso search.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso

from config import THRESHOLD
from pipeline import split_xy, build_pipeline, build_preprocessor
from cv import cross_validate, summarize, tune_alpha, best_alpha

processed = Path("../data/processed")
figures = Path("../output/figures")
tables = Path("../output/tables")
figures.mkdir(parents=True, exist_ok=True)
tables.mkdir(parents=True, exist_ok=True)

## 1. Load the data and lock away the test set

Exactly as in `02_estimate.ipynb`: join the table with the split and keep only the buildings with a fold (0–4).

In [ ]:
data = pd.read_csv(processed / "building_clean.csv")
split = pd.read_csv(processed / "split.csv")
data = data.merge(split, on="EGID", how="inner", validate="one_to_one")

train = data[data["test"] == False].reset_index(drop=True)   # the test set stays closed until task 26
assert train["fold"].notna().all(), "a building without a fold is left"
train["fold"] = train["fold"].astype(int)

X, y, is_house, groups = split_xy(train)
folds = train["fold"].to_numpy()

print("Buildings in cross-validation:", len(train), "| of which houses:", is_house.sum())
print("Groups in more than one fold:", (train.groupby("group_id")["fold"].nunique() > 1).sum())   # expect 0

## 2. The feature sets

Each set is a combination of the three switches of `build_pipeline`. To see what a switch removes, we count the
columns after preprocessing (fitted on fold 1–4, as in the leakage check of task 20).

In [ ]:
FEATURE_SETS = {
    "Full":                        {"heating": True,  "municipality": True,  "expert_rule": True},
    "Full, no expert rule":        {"heating": True,  "municipality": True,  "expert_rule": False},
    "No municipality":             {"heating": True,  "municipality": False, "expert_rule": True},
    "No heating":                  {"heating": False, "municipality": True,  "expert_rule": False},
    "No heating, no municipality": {"heating": False, "municipality": False, "expert_rule": False},
}
REFERENCE = "Full"

check_train = folds != 0
n_columns = {}
rows = []
for label, switches in FEATURE_SETS.items():
    prep = build_preprocessor(**switches).fit(X[check_train], y[check_train])
    names = prep.get_feature_names_out()
    n_columns[label] = len(names)
    rows.append({"feature set": label, **switches, "columns": len(names),
                 "expert-rule column": "expert_rule" in names})
pd.DataFrame(rows).set_index("feature set")

## 3. Cross-validation for every feature set

Linear regression has no setting. For ridge and lasso we try the same values of alpha as in task 20 and take, for
every feature set, the alpha with the highest mean selection measure on houses (smaller MAE on houses if equal).

In [ ]:
RIDGE_ALPHAS = np.logspace(-2, 5, 29)   # the same values as in task 20
LASSO_ALPHAS = np.logspace(-3, 2, 26)
MODELS = {
    "Linear regression": (LinearRegression, None, {}),
    "Ridge": (Ridge, RIDGE_ALPHAS, {}),
    "Lasso": (Lasso, LASSO_ALPHAS, {"max_iter": 10000}),
}

warnings.filterwarnings("ignore", message="Objective did not converge")  # lasso at the smallest alphas, as in task 20

runs = {}   # (feature set, model) -> chosen alpha, per-fold results, summary
for label, switches in FEATURE_SETS.items():
    for name, (model_class, alphas, kwargs) in MODELS.items():
        if alphas is None:
            alpha = np.nan
            make_model = lambda: build_pipeline(model_class(), **switches)
        else:
            tuning = tune_alpha(model_class, alphas, X, y, is_house, folds, model_kwargs=kwargs, **switches)
            alpha = best_alpha(tuning)
            make_model = lambda: build_pipeline(model_class(alpha=alpha, **kwargs), **switches)
        per_fold, _ = cross_validate(make_model, X, y, is_house, folds)
        runs[(label, name)] = {"alpha": alpha, "per_fold": per_fold, "summary": summarize(per_fold)}
    print(label, "| best alpha: ridge", f"{runs[(label, 'Ridge')]['alpha']:.3g}",
          "| lasso", f"{runs[(label, 'Lasso')]['alpha']:.3g}")

## 4. Check: do we get the numbers of task 20?

"Full" and "Full, no expert rule" are the six variants of task 20. Their chosen alpha, selection measure and MAE
must be the same as in `output/tables/task20_chosen.csv`. If they are, this notebook compares like with like.

In [ ]:
def show(summary, group="houses"):
    # the main numbers of a summary table in percent, for one group (same as in 02_estimate)
    s = summary[summary["group"] == group].iloc[0]
    return {
        "top 20%: above 450 (%)": round(s["share_above_top20_mean"] * 100, 1),
        "± SE": round(s["share_above_top20_se"] * 100, 1),
        "top 10%: above 450 (%)": round(s["share_above_top10_mean"] * 100, 1),
        "top 20%: captured (%)": round(s["captured_top20_mean"] * 100, 1),
        "MAE": round(s["mae_mean"], 1),
    }

task20 = pd.read_csv(tables / "task20_chosen.csv")
checked = 0
for label in ["Full", "Full, no expert rule"]:
    for name in MODELS:
        for group in ["houses", "all buildings"]:
            old = task20[(task20["model"] == name) & (task20["expert_rule"] == FEATURE_SETS[label]["expert_rule"])
                         & (task20["group"] == group)].iloc[0]
            new = show(runs[(label, name)]["summary"], group)
            for column, value in new.items():
                assert np.isclose(old[column], value), f"{label}, {name}, {group}: {column} is {value}, task 20 has {old[column]}"
            if name != "Linear regression":
                assert np.isclose(old["alpha"], runs[(label, name)]["alpha"]), f"{label}, {name}: different alpha"
            checked += 1
print("Same numbers as task 20 in all", checked, "rows")

## 5. Overview

Houses first, the main numbers for all buildings next to them. "columns" is the number of columns after preprocessing.

In [ ]:
rows = []
for (label, name), run in runs.items():
    houses, everyone = show(run["summary"], "houses"), show(run["summary"], "all buildings")
    rows.append({
        "feature set": label, "model": name, "columns": n_columns[label],
        "alpha": "–" if pd.isna(run["alpha"]) else f"{run['alpha']:.3g}",
        **houses,
        "all buildings: top 20%: above 450 (%)": everyone["top 20%: above 450 (%)"],
        "all buildings: MAE": everyone["MAE"],
    })
overview = pd.DataFrame(rows)

# baselines of task 18 as references (same folds and measures)
baselines = pd.read_csv(tables / "task18_baselines.csv").query("group == 'houses'").set_index("model")
random_share = (y[is_house] > THRESHOLD).mean() * 100
expert_rule_baseline = baselines.loc["Expert rule", "top 20%: above 450 (%)"]
print(f"References on houses: random choice {random_share:.1f}% | expert-rule baseline {expert_rule_baseline:.1f}%")
overview

In [ ]:
# the selection measure only, one column per model
overview.pivot(index="feature set", columns="model", values="top 20%: above 450 (%)").loc[list(FEATURE_SETS), list(MODELS)]

## 6. Fold by fold: each feature set against the full set

The means hide how much the folds differ. Here every feature set is compared with the full set **in each fold, on the
same houses and with the same model**. With about 47 houses in the top 20% of a fold, one house is about 2 points.
A difference counts only if it has the same sign in most folds and is clearly larger than its standard error.

In [ ]:
def houses_per_fold(label, name):
    # selection measure on houses in percent, one value per fold
    per_fold = runs[(label, name)]["per_fold"]
    return per_fold[per_fold["group"] == "houses"].set_index("fold")["share_above_top20"] * 100

rows, fold_rows = [], []
for name in MODELS:
    full = houses_per_fold(REFERENCE, name)
    for label in FEATURE_SETS:
        own = houses_per_fold(label, name)
        diff = own - full
        for fold in own.index:
            fold_rows.append({"model": name, "feature set": label, "fold": fold,
                              "top 20%: above 450 (%)": round(own[fold], 1),
                              "minus full (points)": round(diff[fold], 1)})
        if label == REFERENCE:
            continue
        rows.append({
            "model": name, "feature set": label,
            "mean difference to full (points)": round(diff.mean(), 1) + 0.0,   # + 0.0 turns -0.0 into 0.0
            "± SE": round(diff.std() / np.sqrt(len(diff)), 1),
            "better in": f"{(diff > 0).sum()} of {len(diff)} folds",
            "worse in": f"{(diff < 0).sum()} of {len(diff)} folds",
        })
per_fold_table = pd.DataFrame(fold_rows)
differences = pd.DataFrame(rows)
differences

## 7. Figure

Share of houses above 450 in the top 20% (mean over the 5 folds, bars = ± one standard error), with random choice
and the expert-rule baseline of task 18 as references.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
labels = list(FEATURE_SETS)
offsets = np.linspace(-0.2, 0.2, len(MODELS))
for offset, name in zip(offsets, MODELS):
    rows_model = overview[overview["model"] == name].set_index("feature set").loc[labels]
    ax.errorbar(np.arange(len(labels)) + offset, rows_model["top 20%: above 450 (%)"], yerr=rows_model["± SE"],
                fmt="o", capsize=3, label=name)
ax.axhline(random_share, color="grey", linestyle="--", linewidth=1.2, label=f"random choice ({random_share:.1f}%)")
ax.axhline(expert_rule_baseline, color="black", linestyle=":", linewidth=1.2,
           label=f"baseline: expert rule ({expert_rule_baseline:.1f}%)")
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels([label.replace(", ", ",\n") for label in labels])
ax.set_ylabel("Houses above 450 in top 20% (%)")
ax.set_title("Feature sets: houses above 450 in the top 20%")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
fig.savefig(figures / "task22_feature_sets.png", dpi=150)
plt.show()

## 8. Save the results

- `output/tables/task22_feature_sets.csv`: the overview of section 5
- `output/tables/task22_per_fold.csv`: the selection measure on houses per fold, with the difference to the full set

In [ ]:
overview.to_csv(tables / "task22_feature_sets.csv", index=False)
per_fold_table.to_csv(tables / "task22_per_fold.csv", index=False)
print("Saved:", len(overview), "rows in task22_feature_sets.csv,", len(per_fold_table), "rows in task22_per_fold.csv")

## 9. What we found

- **No feature set is clearly better or worse than the full set.** Over all 15 combinations the share of houses
  above 450 in the top 20% lies between 52.3% and 55.7%. Every difference to the full set (−1.7 to +1.3 points) is
  smaller than its standard error (1.0 to 4.4 points), and no feature set wins or loses in more than 3 of 5 folds.
- **Without heating information the ranking of houses is as good as with it.** Linear regression 53.2% instead of
  54.0% (−0.9 ± 2.1 points), ridge 55.7% instead of 54.5% (+1.3 ± 3.8), lasso 54.9% instead of 54.0% (+0.9 ± 1.4).
  The expert-rule column also changes nothing (−0.9 to 0.0 points, as already seen in task 20).
- **Without municipality the result is the same** (54.5% for all three models, 0.0 to +0.4 points), with 54 instead
  of 92 columns. For all buildings the measure falls a little, from 56.1–57.1% to 55.4%.
- **The smallest model (size, construction period, building class; 34 columns) is slightly lower and less stable:**
  52.3–53.2%, that is −0.9 to −1.7 points with ± 4.2 to 4.4, and its standard error over the folds doubles (6–7
  instead of about 3 points).
- **All feature sets stay well above the references:** random choice 38.0%, expert-rule baseline 45.1% (task 18).
  Even the models that do not see the heating at all rank houses better than the expert rule, which is built on
  the energy source.
- **The exact values hardly move.** MAE on houses 104.5 to 107.2 in all sets. For all buildings it rises from 75.4
  (full) to 76.6 without heating and 77.2–77.8 for the smallest model, so heating and municipality help the level a
  little, but not the order of the houses.
- **Top 10%:** the sets without heating look better here (54–57% against 43–49%), but a top 10% holds only about
  23 houses per fold, so one house is about 4 points. We do not read anything into it.

**Caution**
- For ridge and lasso, alpha was chosen on the same 5 folds we report, which makes their numbers a bit optimistic,
  for all feature sets equally. Linear regression has no alpha and is the cleanest comparison.
- We compare 5 feature sets and 3 models on about 47 houses per top 20%. With so many comparisons, a few points
  up or down are expected by chance.
- Our data only holds buildings whose heating in the register matches the target years (task 9). For undeclared
  houses the register may be out of date, so the heating information can only be less useful there than here.

**Meaning for the story** (feedback point 2): the worry that today's heating entry does not match the target years
has little weight for our result, because the ranking of houses does not depend on the heating information. Most of
what the models know about houses comes from size, construction period and building class. Which feature set goes
into the final model is decided in task 26; this notebook shows that a model without heating information would be
a defensible choice.